
# RNA-seq Workflow: FASTQ to Quantification and Beyond

## 1. Before sequencing: experimental design

Decisions made here can't be fixed later by analysis.

- **Biological replicates:** at least 3 per condition, more if effects are subtle. Technical replicates matter much less.
- **Sequencing depth:** roughly 20–30 million reads per sample for standard gene-level differential expression; more for isoform analysis or low-expressed genes.
- **Read type:** single-end is enough for basic gene counting; paired-end is better for isoforms, splice junctions, and fusion detection.
- **Library type:** poly-A selection (mRNA only) vs rRNA depletion (also captures non-polyadenylated RNA). **Strandedness** tells you which DNA strand a read came from and must be specified correctly in later steps.
- **Batch effects:** spread conditions evenly across sequencing runs, library prep days, etc., so batch isn't confounded with biology.

## 2. Raw data: FASTQ files

Each read has four lines: an ID, the sequence, a "+" separator, and a quality string. Quality is encoded as a **Phred score**: Q20 means 1% chance of error, Q30 means 0.1%.

## 3. Raw read quality control

**Tools:** FastQC per sample, MultiQC to combine reports.

**What to check:**
- Per-base quality (usually drops toward the 3′ end)
- Adapter content
- GC content distribution (unexpected peaks may mean contamination)
- Overrepresented sequences (adapters, rRNA)
- Duplication levels (high duplication is partly normal in RNA-seq because highly expressed genes produce many identical reads)

## 4. Trimming and filtering

**Tools:** fastp, Trimmomatic, cutadapt, Trim Galore.

Remove adapters and low-quality bases, drop reads that become too short. Modern aligners handle mild quality issues, so aggressive trimming is often unnecessary. Re-run FastQC afterward.

## 5. Alignment (mapping) or pseudo-alignment

This is the step that places reads onto the genome or transcriptome.

**Option A, splice-aware alignment to the genome:**
- **STAR** (fast, memory-hungry) or **HISAT2** (lighter).
- Needs a reference genome (FASTA) and annotation (**GTF/GFF**).
- Output: **SAM/BAM** files (sorted and indexed with samtools).
- "Splice-aware" matters because reads from mRNA span exon–exon junctions.

**Option B, pseudo-alignment to the transcriptome:**
- **Salmon** or **kallisto**: much faster, directly output transcript-level estimates (counts and TPM).
- No BAM files, so less suitable if you need to visualize reads or detect novel transcripts/variants.

**Key concept, multi-mapping reads:** reads that map equally well to several locations (gene families, repeats). Count-based tools usually discard them; Salmon/kallisto distribute them probabilistically.

## 6. Post-alignment QC

**Tools:** samtools flagstat, Qualimap, RSeQC, Picard, MultiQC.

**What to check:**
- **Mapping rate:** typically 70–90%+ uniquely mapped for a good human/mouse sample.
- **Read distribution:** most reads should land in exons; many intronic reads may suggest pre-mRNA or DNA contamination.
- **rRNA fraction:** high values mean poor rRNA removal.
- **Gene body coverage:** strong 3′ bias suggests RNA degradation (check RIN values from library prep).
- **Strandedness:** confirm it matches the library type.

## 7. Quantification

**Tools:** featureCounts or HTSeq-count (from BAM), or Salmon/kallisto output imported with tximport.

Output: a **count matrix** with genes as rows, samples as columns, and raw read counts as values. These raw counts are the input for differential expression.

## 8. Normalization

| Method | Corrects depth | Corrects length | Main use |
|---|---|---|---|
| **CPM** | Yes | No | Filtering low-expressed genes, same-gene comparisons |
| **RPKM / FPKM / CPKM** | Yes | Yes (depth first) | Older; per-sample sums vary |
| **TPM** | Yes | Yes (length first) | Within-sample gene comparison, reporting expression |
| **DESeq2 median-of-ratios** | Yes + composition | No | Differential expression |
| **edgeR TMM** | Yes + composition | No | Differential expression |

**Composition bias** is the key idea behind DESeq2/edgeR normalization: if a few genes are hugely expressed in one sample, they "use up" reads, making all other genes look lower. CPM and TPM don't fix this; median-of-ratios and TMM do.

## 9. Sample-level QC (exploratory analysis)

Before testing anything, look at the samples as a whole:
- **PCA plot** (often on VST or rlog-transformed counts in DESeq2): replicates should cluster together, and conditions should separate.
- **Sample correlation heatmap / hierarchical clustering.**
- **Outlier detection:** a sample clustering with the wrong group may be mislabeled or failed.
- **Batch effects:** if samples cluster by batch rather than condition, include batch in the model design (e.g., `~ batch + condition`).

## 10. Differential expression

**Tools:** DESeq2, edgeR, limma-voom. These model counts with a **negative binomial distribution**, which handles the extra variability (**overdispersion**) seen between biological replicates.

**Key outputs and concepts:**
- **log2 fold change:** magnitude and direction of change. Use LFC shrinkage (e.g., `lfcShrink` in DESeq2) to avoid inflated values for low-count genes.
- **p-value and adjusted p-value (padj / FDR):** with ~20,000 genes tested, raw p-values give many false positives, so use Benjamini–Hochberg FDR (commonly padj < 0.05).
- **Visualization:** volcano plots, MA plots, heatmaps of top genes.

* **Note: DESeq2 or edgeR need raw counts as input , rather than TPM or CPKM**

## 11. Functional interpretation

- **Over-representation analysis:** GO, KEGG, Reactome on a list of significant genes (clusterProfiler, g:Profiler, DAVID).
- **GSEA:** uses the full ranked gene list, so it can detect coordinated small changes that thresholds miss.

## Key concepts checklist

**Data formats:** FASTQ, FASTA, GTF/GFF, SAM/BAM.
**Quality:** Phred score, adapter contamination, duplication, RIN/RNA degradation, mapping rate, rRNA contamination, gene body coverage.
**Library:** single vs paired-end, stranded vs unstranded, poly-A vs rRNA depletion, sequencing depth.
**Mapping:** splice-aware alignment, multi-mapping reads, genome vs transcriptome reference, pseudo-alignment.
**Normalization:** library size, gene length bias, composition bias, CPM vs RPKM/FPKM vs TPM vs DESeq2/TMM.
**Statistics:** biological replicates, negative binomial model, dispersion, fold change, multiple testing correction (FDR), batch effects and design formulas.

## Common pitfalls

- Using TPM/FPKM as input to DESeq2 or edgeR (use raw counts).
- Setting strandedness wrong in featureCounts/HTSeq, which can lose most of your counts.
- Mismatched genome and annotation versions (e.g., GRCh38 genome with a GRCh37 GTF).
- Too few replicates, or batch fully confounded with condition.
- Ignoring PCA/QC and running DE on a dataset with a failed or swapped sample.

A typical tool chain to start with: **FastQC → fastp → STAR → featureCounts → DESeq2 → clusterProfiler**, or the faster route **FastQC → fastp → Salmon → tximport → DESeq2**.